#Initialization

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab

Mounted at /content/drive
/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab


In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "123"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # must be before torch import

In [ ]:
import math, random, hashlib, copy
import pandas as pd
import numpy as np

from pathlib import Path
from typing  import Tuple, List
from PIL     import Image, ImageEnhance

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from sklearn.metrics import fbeta_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [ ]:
print("Device:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("PyTorch version:", torch.__version__)

Device: Tesla T4
CUDA version: 12.8
cuDNN version: 91900
PyTorch version: 2.11.0+cu128


In [ ]:
sc1_result = []

#Data Loading

##Sc.1.a. Data

In [ ]:
val_sc1a_df   = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc1a/val_sc1a.xlsx")
test_sc1a_df  = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc1a/test_sc1a.xlsx")

val_sc1b_df   = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc1b/val_sc1b.xlsx")
test_sc1b_df  = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc1b/test_sc1b.xlsx")


#Pre-Processing Set

In [ ]:
SEED = 123
EPOCHS = 20

In [ ]:
# Transformation

IMAGE_SIZE = 224

imagenet_norm = transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])

eval_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    imagenet_norm,
])

In [ ]:
# Dataset Loader

BATCH_SIZE = 32
NUM_WORKERS = 2

class FaceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        sample = self.dataframe.iloc[idx]
        image  = Image.open(sample["path"]).convert("RGB")
        image  = self.transform(image) if self.transform else transforms.ToTensor()(image)
        label  = int(sample["label"])
        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "child_id": str(sample["child_id"]),
            "path": str(sample["path"]),
        }

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(dataframe, transform, shuffle):
    ds = FaceDataset(dataframe, transform=transform)
    g  = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=g,
        worker_init_fn=seed_worker,  # ← ADD THIS
    )

#Model Testing

In [ ]:
# Prediction: Function

def collect_predictions(model, loader):
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in loader:
            xb = batch["image"].to(device, non_blocking=True)
            yb = batch["label"].cpu().numpy().astype(int)
            logits = model(xb)
            probs  = torch.softmax(logits, dim=1).detach().cpu().numpy()
            logits_np = logits.detach().cpu().numpy()

            for i in range(len(yb)):
                rows.append({
                    "child_id": batch["child_id"][i],
                    "path": batch["path"][i],
                    "label": int(yb[i]),
                    "logit0": float(logits_np[i, 0]),
                    "logit1": float(logits_np[i, 1]),
                    "prob0": float(probs[i, 0]),
                    "prob1": float(probs[i, 1])
                })
    return pd.DataFrame(rows)

In [ ]:
#Evaluation Function

def safe_div(num, den):
    return float(num) / float(den) if den else 0.0

def metric_bundle(y_true, prob1, threshold):
    y_true = np.asarray(y_true).astype(int)
    prob1 = np.asarray(prob1).astype(float)
    y_pred = (prob1 >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    accuracy = safe_div(tn + tp, len(y_true))
    precision = safe_div(tp, tp + fp)
    sensitivity = safe_div(tp, tp + fn)   # recall for stunting
    specificity = safe_div(tn, tn + fp)
    f1 = safe_div(2 * precision * sensitivity, precision + sensitivity)
    beta2 = 2.0
    f2 = safe_div((1 + beta2**2) * precision * sensitivity, (beta2**2) * precision + sensitivity)
    bal_acc = 0.5 * (sensitivity + specificity)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1,
        "f2": f2,
        "balanced_accuracy": bal_acc,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


In [ ]:
# Model - pre-trained EfficientNet B0

def create_model(num_classes=2):
    weights = EfficientNet_B0_Weights.DEFAULT
    model   = efficientnet_b0(weights=weights)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model

##Sc.1.a. EfficientNet-B0

###Sc.1.a. EfficientNet-B0 with SGD

In [ ]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 173MB/s]


In [ ]:
# Load Dataset
dl_val_sc1a   = make_loader(val_sc1a_df, eval_tfms, shuffle=False)
dl_test_sc1a  = make_loader(test_sc1a_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc1a_df)} | test={len(test_sc1a_df)}")


val=400 | test=400


In [ ]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1a_sgd.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc1a)
test_predict  = collect_predictions(model, dl_test_sc1a)

####Sc.1.a. Efficient-B0 with SGD Validation Data

In [ ]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 1.0,
 'precision': 1.0,
 'sensitivity': 1.0,
 'specificity': 1.0,
 'f1': 1.0,
 'f2': 1.0,
 'balanced_accuracy': 1.0,
 'TN': 200,
 'FP': 0,
 'FN': 0,
 'TP': 200}

In [ ]:
sc1_result.append([
    "sc1a-efficientnet-b0-sgd",
    "sc1a", "efficientnet-b0", "sgd",
    "validation",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

####Sc.1.a. Efficient-B0 with SGD Test Data

In [ ]:
# Evaluation for Test data
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=0.5)
result

test evaluation:


{'accuracy': 1.0,
 'precision': 1.0,
 'sensitivity': 1.0,
 'specificity': 1.0,
 'f1': 1.0,
 'f2': 1.0,
 'balanced_accuracy': 1.0,
 'TN': 200,
 'FP': 0,
 'FN': 0,
 'TP': 200}

In [ ]:
sc1_result.append([
    "sc1a-efficientnet-b0-sgd",
    "sc1a", "efficientnet-b0", "sgd",
    "test",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

###Sc.1.a. EfficientNet-B0 with Adam

In [ ]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda


In [ ]:
# Load Dataset
dl_val_sc1a   = make_loader(val_sc1a_df, eval_tfms, shuffle=False)
dl_test_sc1a  = make_loader(test_sc1a_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc1a_df)} | test={len(test_sc1a_df)}")


val=400 | test=400


In [ ]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1a_adam.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc1a)
test_predict  = collect_predictions(model, dl_test_sc1a)

####Sc.1.a. Efficient-B0 with Adam Validation Data

In [ ]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 1.0,
 'precision': 1.0,
 'sensitivity': 1.0,
 'specificity': 1.0,
 'f1': 1.0,
 'f2': 1.0,
 'balanced_accuracy': 1.0,
 'TN': 200,
 'FP': 0,
 'FN': 0,
 'TP': 200}

In [ ]:
sc1_result.append([
    "sc1a-efficientnet-b0-adam",
    "sc1a", "efficientnet-b0", "adam",
    "validation",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

####Sc.1.a. Efficient-B0 with Adam Test Data

In [ ]:
# Evaluation for Test data
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=0.5)
result

test evaluation:


{'accuracy': 1.0,
 'precision': 1.0,
 'sensitivity': 1.0,
 'specificity': 1.0,
 'f1': 1.0,
 'f2': 1.0,
 'balanced_accuracy': 1.0,
 'TN': 200,
 'FP': 0,
 'FN': 0,
 'TP': 200}

In [ ]:
sc1_result.append([
    "sc1a-efficientnet-b0-adam",
    "sc1a", "efficientnet-b0", "adam",
    "test",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

###Sc.1.a. EfficientNet-B0 with AdamW

In [ ]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda


In [ ]:
# Load Dataset
dl_val_sc1a   = make_loader(val_sc1a_df, eval_tfms, shuffle=False)
dl_test_sc1a  = make_loader(test_sc1a_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc1a_df)} | test={len(test_sc1a_df)}")


val=400 | test=400


In [ ]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1a_adamw.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc1a)
test_predict  = collect_predictions(model, dl_test_sc1a)

####Sc.1.a. Efficient-B0 with AdamW Validation Data

In [ ]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 1.0,
 'precision': 1.0,
 'sensitivity': 1.0,
 'specificity': 1.0,
 'f1': 1.0,
 'f2': 1.0,
 'balanced_accuracy': 1.0,
 'TN': 200,
 'FP': 0,
 'FN': 0,
 'TP': 200}

In [ ]:
sc1_result.append([
    "sc1a-efficientnet-b0-adamw",
    "sc1a", "efficientnet-b0", "adamw",
    "validation",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

####Sc.1.a. Efficient-B0 with AdamW Test Data

In [ ]:
# Evaluation for Test data
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=0.5)
result

test evaluation:


{'accuracy': 1.0,
 'precision': 1.0,
 'sensitivity': 1.0,
 'specificity': 1.0,
 'f1': 1.0,
 'f2': 1.0,
 'balanced_accuracy': 1.0,
 'TN': 200,
 'FP': 0,
 'FN': 0,
 'TP': 200}

In [ ]:
sc1_result.append([
    "sc1a-efficientnet-b0-adamw",
    "sc1a", "efficientnet-b0", "adamw",
    "test",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

##Sc.1.b. EfficientNet-B0

###Sc.1.b. EfficientNet-B0 with SGD

In [ ]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda


In [ ]:
# Load Dataset
dl_val_sc1b   = make_loader(val_sc1b_df, eval_tfms, shuffle=False)
dl_test_sc1b  = make_loader(test_sc1b_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc1b_df)} | test={len(test_sc1b_df)}")


val=400 | test=400


In [ ]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1b_sgd.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc1b)
test_predict  = collect_predictions(model, dl_test_sc1b)

####Sc.1.b. Efficient-B0 with SGD Validation Data

In [ ]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 0.5925,
 'precision': 0.5583596214511041,
 'sensitivity': 0.885,
 'specificity': 0.3,
 'f1': 0.6847195357833655,
 'f2': 0.7923008057296329,
 'balanced_accuracy': 0.5925,
 'TN': 60,
 'FP': 140,
 'FN': 23,
 'TP': 177}

In [ ]:
sc1_result.append([
    "sc1b-efficientnet-b0-sgd",
    "sc1b", "efficientnet-b0", "sgd",
    "validation",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

####Sc.1.b. Efficient-B0 with SGD Test Data

In [ ]:
# Evaluation for Test data
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=0.5)
result

test evaluation:


{'accuracy': 0.6775,
 'precision': 0.6898395721925134,
 'sensitivity': 0.645,
 'specificity': 0.71,
 'f1': 0.6666666666666666,
 'f2': 0.6534954407294833,
 'balanced_accuracy': 0.6775,
 'TN': 142,
 'FP': 58,
 'FN': 71,
 'TP': 129}

In [ ]:
sc1_result.append([
    "sc1b-efficientnet-b0-sgd",
    "sc1b", "efficientnet-b0", "sgd",
    "test",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

###Sc.1.b. EfficientNet-B0 with Adam

In [ ]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda


In [ ]:
# Load Dataset
dl_val_sc1b   = make_loader(val_sc1b_df, eval_tfms, shuffle=False)
dl_test_sc1b  = make_loader(test_sc1b_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc1b_df)} | test={len(test_sc1b_df)}")


val=400 | test=400


In [ ]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1b_adam.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc1b)
test_predict  = collect_predictions(model, dl_test_sc1b)

####Sc.1.b. Efficient-B0 with Adam Validation Data

In [ ]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 0.625,
 'precision': 0.5757575757575758,
 'sensitivity': 0.95,
 'specificity': 0.3,
 'f1': 0.7169811320754716,
 'f2': 0.8407079646017699,
 'balanced_accuracy': 0.625,
 'TN': 60,
 'FP': 140,
 'FN': 10,
 'TP': 190}

In [ ]:
sc1_result.append([
    "sc1b-efficientnet-b0-adam",
    "sc1b", "efficientnet-b0", "adam",
    "validation",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

####Sc.1.b. Efficient-B0 with Adam Test Data

In [ ]:
# Evaluation for Test data
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=0.5)
result

test evaluation:


{'accuracy': 0.735,
 'precision': 0.7079646017699115,
 'sensitivity': 0.8,
 'specificity': 0.67,
 'f1': 0.7511737089201878,
 'f2': 0.7797270955165694,
 'balanced_accuracy': 0.7350000000000001,
 'TN': 134,
 'FP': 66,
 'FN': 40,
 'TP': 160}

In [ ]:
sc1_result.append([
    "sc1b-efficientnet-b0-adam",
    "sc1b", "efficientnet-b0", "adam",
    "test",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

###Sc.1.b. EfficientNet-B0 with AdamW

In [ ]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda


In [ ]:
# Load Dataset
dl_val_sc1b   = make_loader(val_sc1b_df, eval_tfms, shuffle=False)
dl_test_sc1b  = make_loader(test_sc1b_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc1b_df)} | test={len(test_sc1b_df)}")


val=400 | test=400


In [ ]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1b_adamw.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc1b)
test_predict  = collect_predictions(model, dl_test_sc1b)

####Sc.1.b. Efficient-B0 with AdamW Validation Data

In [ ]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 0.625,
 'precision': 0.5757575757575758,
 'sensitivity': 0.95,
 'specificity': 0.3,
 'f1': 0.7169811320754716,
 'f2': 0.8407079646017699,
 'balanced_accuracy': 0.625,
 'TN': 60,
 'FP': 140,
 'FN': 10,
 'TP': 190}

In [ ]:
sc1_result.append([
    "sc1b-efficientnet-b0-adamw",
    "sc1b", "efficientnet-b0", "adamw",
    "validation",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

####Sc.1.b. Efficient-B0 with AdamW Test Data

In [ ]:
# Evaluation for Test data
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=0.5)
result

test evaluation:


{'accuracy': 0.7325,
 'precision': 0.7048458149779736,
 'sensitivity': 0.8,
 'specificity': 0.665,
 'f1': 0.7494145199063232,
 'f2': 0.7789678675754625,
 'balanced_accuracy': 0.7325,
 'TN': 133,
 'FP': 67,
 'FN': 40,
 'TP': 160}

In [ ]:
sc1_result.append([
    "sc1b-efficientnet-b0-adamw",
    "sc1b", "efficientnet-b0", "adamw",
    "test",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

#Result

In [ ]:
sc1_result_df = pd.DataFrame(sc1_result, columns=['model_name',
                                                  'scheme', 'model', 'optimizer',
                                                  'evaluation',
                                                  'accuracy',
                                                  'precision',
                                                  'sensitivity',
                                                  'specificity',
                                                  'f1',
                                                  'f2',
                                                  'TN',
                                                  'FP',
                                                  'FN',
                                                  'TP',
                                                  ])
display(sc1_result_df)

base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/result/"
sc1_result_df.to_excel(base + 'sc1_result.xlsx', index=False)

,model_name,scheme,model,optimizer,evaluation,accuracy,precision,sensitivity,specificity,f1,f2,TN,FP,FN,TP
0,sc1a-efficientnet-b0-sgd,sc1a,efficientnet-b0,sgd,validation,1.0000,1.000000,1.000,1.000,1.000000,1.000000,200,0,0,200
1,sc1a-efficientnet-b0-sgd,sc1a,efficientnet-b0,sgd,test,1.0000,1.000000,1.000,1.000,1.000000,1.000000,200,0,0,200
2,sc1a-efficientnet-b0-adam,sc1a,efficientnet-b0,adam,validation,1.0000,1.000000,1.000,1.000,1.000000,1.000000,200,0,0,200
3,sc1a-efficientnet-b0-adam,sc1a,efficientnet-b0,adam,test,1.0000,1.000000,1.000,1.000,1.000000,1.000000,200,0,0,200
4,sc1a-efficientnet-b0-adamw,sc1a,efficientnet-b0,adamw,validation,1.0000,1.000000,1.000,1.000,1.000000,1.000000,200,0,0,200
5,sc1a-efficientnet-b0-adamw,sc1a,efficientnet-b0,adamw,test,1.0000,1.000000,1.000,1.000,1.000000,1.000000,200,0,0,200
6,sc1b-efficientnet-b0-sgd,sc1b,efficientnet-b0,sgd,validation,0.5925,0.558360,0.885,0.300,0.684720,0.792301,60,140,23,177
7,sc1b-efficientnet-b0-sgd,sc1b,efficientnet-b0,sgd,test,0.6775,0.689840,0.645,0.710,0.666667,0.653495,142,58,71,129
8,sc1b-efficientnet-b0-adam,sc1b,efficientnet-b0,adam,validation,0.6250,0.575758,0.950,0.300,0.716981,0.840708,60,140,10,190
9,sc1b-efficientnet-b0-adam,sc1b,efficientnet-b0,adam,test,0.7350,0.707965,0.800,0.670,0.751174,0.779727,134,66,40,160
